# Phase 5 — Embedding Polarity-Only: Drop neg2 Experiment (SemEval 2014)

**Goal:** Verify root cause of polarity blindness by removing neg2 (category negative) from contrastive training. Only neg1 (polarity negative) remains — forces model to learn polarity separation.

**Hypothesis:** With neg2 removed, margin_neg1 should increase significantly (vs ~0.001 with neg2).

**Two-stage training (same as v4, but no neg2):**
1. **Stage 1:** Train on polarity-only triplets (`contrastive_triplets_polonly.jsonl`)
2. **Hard negative mining:** Build FAISS index from Stage 1 → mine hard negatives (no neg2)
3. **Stage 2:** Train on hard polarity-only triplets
4. **Build FAISS index** from Stage 2 model (final)

**Dataset:** SemEval 2014 Task 4 Restaurant (5 categories, 3 polarities)

**Input:** `lcminhc/semeval-2014-absa-restaurant` (processed data)

**Output:** Upload `/kaggle/working/outputs_p5_embed_v4/` as Kaggle dataset `p5-embed-v4`
- `embedding_v4_s1_best.pt` (Stage 1 checkpoint)
- `embedding_v4_s2_best.pt` (Stage 2 checkpoint — final)
- `embedding_best.pt` (= Stage 2, compatible name for NB2/NB3)
- `train.faiss`, `metadata.json`, `train_vectors.npy` (FAISS index)
- Training logs

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os
import sys
import json
import shutil
import glob

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
# Wire dataset
KAGGLE_INPUT = None
for candidate in ['/kaggle/input/semeval-2014-absa-restaurant',
                  '/kaggle/input/datasets/lcminhc/semeval-2014-absa-restaurant']:
    if os.path.exists(candidate):
        KAGGLE_INPUT = candidate
        break
assert KAGGLE_INPUT, 'Dataset semeval-2014-absa-restaurant not found'
print(f'Input: {KAGGLE_INPUT}')
print('Files:', os.listdir(KAGGLE_INPUT))

# Copy processed data for embedding training
os.makedirs('data/processed', exist_ok=True)
for f in ['contrastive_triplets.jsonl', 'classification.jsonl',
          'sentiment_records.jsonl']:
    src = f'{KAGGLE_INPUT}/{f}'
    dst = f'data/processed/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        with open(dst) as fp:
            n = sum(1 for _ in fp)
        print(f'{f}: {n} records')
    else:
        print(f'WARNING: {f} not found in dataset')

# Generate polarity-only triplets (strip neg2 fields from existing triplets)
import json as _json
polonly_path = 'data/processed/contrastive_triplets_polonly.jsonl'
neg2_keys = {'neg2_id', 'neg2_sentence', 'neg2_aspect', 'neg2_polarity'}
with open('data/processed/contrastive_triplets.jsonl') as fin, \
     open(polonly_path, 'w') as fout:
    count = 0
    for line in fin:
        rec = _json.loads(line)
        for k in neg2_keys:
            rec.pop(k, None)
        fout.write(_json.dumps(rec) + '\n')
        count += 1
print(f'contrastive_triplets_polonly.jsonl: {count} records (neg2 stripped)')

In [ ]:
import torch
import gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 1. Stage 1 — Train on Polarity-Only Triplets

Config: `embedding_2014_polonly_s1.yaml`
- triplets: `contrastive_triplets_polonly.jsonl` (neg2 stripped)
- tau=0.07, batch=128, epochs=15, patience=5
- gradient_checkpointing=true
- **Key metric:** margin_neg1 — should increase significantly vs v4 baseline (~0.001)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

!python scripts/02_train_embedding.py --config configs/embedding_2014_polonly_s1.yaml

In [ ]:
# Stage 1 training log
log_path = 'logs/embedding_2014_polonly_s1.jsonl'
print('=== Embedding Stage 1 (Polarity-Only) Training Log ===')
if os.path.exists(log_path):
    print(f'{"Ep":<4} {"Loss":<8} {"R@1":<6} {"R@3":<6} {"R@5":<6} {"intra":<7} {"inter":<7} {"ratio":<7} {"m1":<7}')
    print('-' * 58)
    with open(log_path) as f:
        for line in f:
            r = json.loads(line)
            print(f"{r['epoch']:<4} {r['train_loss']:<8.4f} "
                  f"{r.get('recall@1',0):<6.3f} {r.get('recall@3',0):<6.3f} {r.get('recall@5',0):<6.3f} "
                  f"{r.get('intra_sim',0):<7.4f} {r.get('inter_sim',0):<7.4f} {r.get('intra_inter_ratio',0):<7.3f} "
                  f"{r.get('margin_neg1',0):<7.3f}")
else:
    print('No log found.')

ckpt = 'checkpoints/embedding_2014_polonly_s1/best.pt'
if os.path.exists(ckpt):
    print(f'\nStage 1 checkpoint: {os.path.getsize(ckpt)/1e6:.1f} MB')
else:
    print('\nERROR: Stage 1 checkpoint not found!')

In [ ]:
# Save Stage 1 checkpoint immediately
output_dir = '/kaggle/working/outputs_p5_embed_v4'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f'{output_dir}/logs', exist_ok=True)

if os.path.exists('checkpoints/embedding_2014_polonly_s1/best.pt'):
    shutil.copy('checkpoints/embedding_2014_polonly_s1/best.pt',
                f'{output_dir}/embedding_v4_s1_best.pt')
    print('Stage 1 ckpt saved')
if os.path.exists('logs/embedding_2014_polonly_s1.jsonl'):
    shutil.copy('logs/embedding_2014_polonly_s1.jsonl',
                f'{output_dir}/logs/embedding_2014_polonly_s1.jsonl')
    print('Stage 1 log saved')

## 2. Hard Negative Mining (Polarity-Only)

Use Stage 1 model to encode all train records → mine hard negatives (neg1 only, no neg2).

In [ ]:
gc.collect()
torch.cuda.empty_cache()

!python scripts/build_hard_triplets.py \
    --embedding_ckpt checkpoints/embedding_2014_polonly_s1/best.pt \
    --cls_path data/processed/classification.jsonl \
    --out_path data/processed/hard_contrastive_triplets_polonly.jsonl \
    --no_neg2

In [ ]:
# Verify hard triplets
hard_path = 'data/processed/hard_contrastive_triplets_polonly.jsonl'
if os.path.exists(hard_path):
    with open(hard_path) as f:
        lines = f.readlines()
    print(f'Hard triplets (polarity-only): {len(lines)}')
    sample = json.loads(lines[0])
    print(f'Keys: {list(sample.keys())}')
    assert 'neg2_id' not in sample, 'neg2 should not be present!'
    if 'pos_sim' in sample:
        print(f'Sample pos_sim={sample["pos_sim"]:.4f}, neg1_sim={sample["neg1_sim"]:.4f}')
else:
    print('ERROR: hard triplets not generated!')

## 3. Stage 2 — Train on Hard Polarity-Only Triplets

Config: `embedding_2014_polonly_s2.yaml`
- triplets: `hard_contrastive_triplets_polonly.jsonl`
- tau=0.07, batch=128, epochs=15, patience=5

In [ ]:
gc.collect()
torch.cuda.empty_cache()

!python scripts/02_train_embedding.py --config configs/embedding_2014_polonly_s2.yaml

In [ ]:
# Stage 2 training log
log_path = 'logs/embedding_2014_polonly_s2.jsonl'
print('=== Embedding Stage 2 (Polarity-Only) Training Log ===')
if os.path.exists(log_path):
    print(f'{"Ep":<4} {"Loss":<8} {"R@1":<6} {"R@3":<6} {"R@5":<6} {"intra":<7} {"inter":<7} {"ratio":<7} {"m1":<7}')
    print('-' * 58)
    with open(log_path) as f:
        for line in f:
            r = json.loads(line)
            print(f"{r['epoch']:<4} {r['train_loss']:<8.4f} "
                  f"{r.get('recall@1',0):<6.3f} {r.get('recall@3',0):<6.3f} {r.get('recall@5',0):<6.3f} "
                  f"{r.get('intra_sim',0):<7.4f} {r.get('inter_sim',0):<7.4f} {r.get('intra_inter_ratio',0):<7.3f} "
                  f"{r.get('margin_neg1',0):<7.3f}")
else:
    print('No log found.')

ckpt = 'checkpoints/embedding_2014_polonly_s2/best.pt'
if os.path.exists(ckpt):
    print(f'\nStage 2 checkpoint: {os.path.getsize(ckpt)/1e6:.1f} MB')
else:
    print('\nERROR: Stage 2 checkpoint not found!')

## 4. Build FAISS Index (from Stage 2 model)

Index `sentiment_records.jsonl` (train split) for retrieval.

In [ ]:
gc.collect()
torch.cuda.empty_cache()

!python scripts/03_build_index.py \
    --embedding_ckpt checkpoints/embedding_2014_polonly_s2/best.pt \
    --input data/processed/sentiment_records.jsonl \
    --out_dir indexes/

In [ ]:
# Verify index
for f in ['train.faiss', 'train_metadata.jsonl', 'train_vectors.npy']:
    path = f'indexes/{f}'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'{f}: {size_mb:.2f} MB')
    else:
        print(f'MISSING: {f}')

## 5. Embedding Quality Check

Quick sanity check: polarity match rate and cosine distribution.

In [ ]:
import numpy as np

vectors = np.load('indexes/train_vectors.npy')
with open('indexes/train_metadata.jsonl') as f:
    metadata = [json.loads(line) for line in f]

print(f'Vectors: {vectors.shape}')
print(f'Metadata: {len(metadata)} records')

# Random pair cosine similarity
rng = np.random.default_rng(42)
n = len(vectors)
idx = rng.integers(0, n, size=(500, 2))
random_sims = np.array([float(vectors[a] @ vectors[b]) for a, b in idx if a != b])
print(f'\nRandom pair cosine: mean={random_sims.mean():.4f}, std={random_sims.std():.4f}')
print(f'  min={random_sims.min():.4f}, max={random_sims.max():.4f}')

# Polarity + Category match rate for top-k
import faiss
index = faiss.read_index('indexes/train.faiss')

k = 5
D, I = index.search(vectors, k + 1)  # +1 for self

pol_counts = {pol: [] for pol in ['positive', 'negative', 'neutral']}
cat_counts = {}
pol_matches_total = 0
cat_matches_total = 0
total = 0

for i in range(n):
    query_pol = metadata[i]['polarity']
    query_cat = metadata[i]['aspect_category']
    neighbors = [j for j in I[i] if j != i][:k]

    pol_m = sum(1 for j in neighbors if metadata[j]['polarity'] == query_pol)
    cat_m = sum(1 for j in neighbors if metadata[j]['aspect_category'] == query_cat)

    pol_counts[query_pol].append(pol_m / k)
    cat_counts.setdefault(query_cat, []).append(cat_m / k)

    pol_matches_total += pol_m
    cat_matches_total += cat_m
    total += len(neighbors)

print(f'\n=== Polarity match rate (top-{k}) ===')
print(f'  Overall: {pol_matches_total/total:.1%}')
for pol in ['positive', 'negative', 'neutral']:
    if pol_counts[pol]:
        rates = np.array(pol_counts[pol])
        print(f'  {pol}: {rates.mean():.1%} (n={len(rates)})')

print(f'\n=== Category match rate (top-{k}) ===')
print(f'  Overall: {cat_matches_total/total:.1%}')
for cat in sorted(cat_counts.keys()):
    rates = np.array(cat_counts[cat])
    print(f'  {cat}: {rates.mean():.1%} (n={len(rates)})')

print(f'\n=== Baseline comparison ===')
print(f'  V4 (with neg2): pol_match@3 ~58%, cat_match@3 ~99.8%')
print(f'  Polonly (no neg2): pol_match@{k} = {pol_matches_total/total:.1%}, cat_match@{k} = {cat_matches_total/total:.1%}')

## 6. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p5_embed_v4'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f'{output_dir}/logs', exist_ok=True)

# Embedding checkpoints
for stage, ckpt_dir in [('s1', 'embedding_2014_polonly_s1'), ('s2', 'embedding_2014_polonly_s2')]:
    src = f'checkpoints/{ckpt_dir}/best.pt'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/embedding_v4_{stage}_best.pt')
        print(f'embedding_v4_{stage}_best.pt: {os.path.getsize(src)/1e6:.1f} MB')
    else:
        print(f'WARNING: {src} not found')

# Also save as embedding_best.pt (compatible name for downstream NB2/NB3)
s2_ckpt = 'checkpoints/embedding_2014_polonly_s2/best.pt'
if os.path.exists(s2_ckpt):
    shutil.copy(s2_ckpt, f'{output_dir}/embedding_best.pt')
    print('embedding_best.pt (= s2 final) saved')

# FAISS index
for f in ['train.faiss', 'metadata.json', 'train_vectors.npy']:
    src = f'indexes/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/{f}')
        print(f'{f}: {os.path.getsize(src)/1e6:.2f} MB')

# Hard triplets (for reference)
hard_path = 'data/processed/hard_contrastive_triplets_polonly.jsonl'
if os.path.exists(hard_path):
    shutil.copy(hard_path, f'{output_dir}/hard_contrastive_triplets_polonly.jsonl')
    print('hard_contrastive_triplets_polonly.jsonl saved')

# Logs
for log in ['embedding_2014_polonly_s1.jsonl', 'embedding_2014_polonly_s2.jsonl']:
    src = f'logs/{log}'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/logs/{log}')

print(f'\n=== Output contents ===')
for root, dirs, files in os.walk(output_dir):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1e6
        rel = os.path.relpath(path, output_dir)
        print(f'  {rel}: {size:.2f} MB')

print(f'\nUpload as Kaggle dataset: p5-embed-v4')

In [ ]:
shutil.make_archive('/kaggle/working/outputs_p5_embed_v4_backup', 'zip',
                    '/kaggle/working', 'outputs_p5_embed_v4')
size_mb = os.path.getsize('/kaggle/working/outputs_p5_embed_v4_backup.zip') / 1e6
print(f'Backup zip: {size_mb:.1f} MB')